# N₂ full-valence CAS(10,8) with atomically aligned IBO final orbitals

This notebook runs a full-valence CAS(10,8) CASSCF calculation on N₂ at 2.0 Å in the cc-pVDZ basis and exercises `final_orbitals="ibo_atomic"`. Point-group symmetry is explicitly disabled because IBO final orbitals are available only in C1 symmetry.

The checks below verify that the final IBO transformation leaves every inactive orbital unchanged, applies a nontrivial unitary rotation only within the active space, aligns the rotationally free transverse p orbitals with the global Cartesian axes, and preserves MO orthonormality. The final cell writes cube files for every optimized orbital.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("FORTE_NUM_THREADS_OVERRIDE", "1")

import numpy as np

from forte2 import MCOptimizer, RHF, CISolver, State, System, write_orbital_cubes
from forte2.orbitals.iao import IBO
from forte2.system.basis_utils import BasisInfo

forte2: using 1 thread for parallel sections
[mods_manager] loading mod determinant_printing from /Users/fevange/.forte2/mods
[mods_manager] failed to load mod determinant_printing from /Users/fevange/.forte2/mods: partially initialized module 'forte2' from '/Users/fevange/Source/forte2-dev-2/forte2/__init__.py' has no attribute 'Determinant' (most likely due to a circular import)


## Molecular system and reference

The two N 1s-like orbitals are kept doubly occupied. The following eight orbitals form the full-valence active space, leaving ten active electrons after the four core electrons are removed from N₂'s fourteen electrons.

In [2]:
bond_length_angstrom = 4.0
xyz = f"""
N  0.0  0.0  0.0
N  0.0  0.0  {bond_length_angstrom}
"""

system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    cholesky_tol=1.0e-10,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

assert system.point_group.upper() == "C1"
print(f"Point group: {system.point_group}")
print(f"Number of cc-pVDZ orbitals: {system.nmo}")

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   N   0.00000000   0.00000000   0.00000000
   N   0.00000000   0.00000000   7.55890450
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 1.713e+00
  Min eigenvalue: 1.849e-01
  Condition number: 9.265e+00
  Inverse condition number: 1.079e-01
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 1.849e-01
Point group: C1
Number of cc-pVDZ orbitals: 28


## Full-valence CAS(10,8) CASSCF with atomically aligned IBO final orbitals

`CISolver` supplies full CI in the eight-orbital valence space. `MCOptimizer` optimizes the orbitals, localizes only the active orbitals with IBO, and then aligns rotationally free shell blocks with canonical atomic IAOs.

In [3]:
singlet = State(system=system, multiplicity=1, ms=0.0)
cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
)
casscf = MCOptimizer(
    cas_solver,
    final_orbitals="ibo_atomic",
)(rhf)
casscf.run()

assert casscf.converged
assert casscf.final_orbitals == "ibo_atomic"
assert casscf.mo_space.nactv == 8
np.testing.assert_allclose(casscf.E, -108.7939008973, atol=1.0e-8)
print(f"CAS(10,8) IBO-CASSCF energy: {casscf.E:.12f} Eh")

Number of electrons: 14
Number of alpha electrons: 7
Number of beta electrons: 7
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: None
Energy convergence criterion: 1.000000e-12
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Building B tensor using Cholesky decomposition
Temporary memory requirement for 4-index integrals: 0.00 GB
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 28
Number of auxiliary basis functions: 204
Iter               Energy           ΔE       ||ΔD||  ||AO grad||      <S^2>  DIIS
---------------------------------------------------------------------------------
   1    -107.868067784830  -8.2477e-03   9.1058e-02   1.6313e-01    0.00000     S
   2    -107.868549500940  -4.8172e-04   2.5759e-02   2.7097e-02    0.00000   S/E
   3    -107.868562358883  -1.2858e-05   4.3665e-03   3.8921e-03    0.0

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=1e-08

Mismatched elements: 1 / 1 (100%)
Max absolute difference among violations: 0.01697881
Max relative difference among violations: 0.00015606
 ACTUAL: array(-108.776922)
 DESIRED: array(-108.793901)

## Validate the final-orbital transformation

`casscf._C` is the converged, contiguous-order coefficient matrix immediately before the final-orbital transformation. Comparing it with `casscf.mos.C[0]` lets us test the scope and character of the IBO rotation. Because the N–N bond is the z axis, each rotationally free transverse pair should also align with the ordered global 2px and 2py IAOs on its atom.

In [ ]:
mo_space = casscf.mo_space
C_before_final = casscf._C[:, mo_space.contig_to_orig]
C_ibo = casscf.mos.C[0]
S = system.ints_overlap()

active = np.asarray(mo_space.active_indices, dtype=int)
inactive = np.asarray(
    sorted(set(range(mo_space.nmo)) - set(mo_space.active_indices)),
    dtype=int,
)

# The IBO final-orbital step must not modify core or virtual orbitals.
np.testing.assert_array_equal(C_ibo[:, inactive], C_before_final[:, inactive])

# Its action inside the active space must be unitary and nontrivial.
U_active = C_before_final[:, active].T @ S @ C_ibo[:, active]
np.testing.assert_allclose(U_active.T @ U_active, np.eye(8), atol=1.0e-10)
assert not np.allclose(np.abs(U_active), np.eye(8), atol=1.0e-3)

# The full final MO set remains orthonormal.
np.testing.assert_allclose(C_ibo.T @ S @ C_ibo, np.eye(mo_space.nmo), atol=1.0e-10)

# Reconstruct the IBO analysis and verify the automatic Cartesian gauge.
ibo_analysis = IBO(system, C_before_final[:, active])
ibo_analysis.align_to_atomic_orbitals()
np.testing.assert_allclose(ibo_analysis.C_ibo, C_ibo[:, active], atol=1.0e-10)
C_ibo_iao = ibo_analysis.C_iao.T @ S @ C_ibo[:, active]
minao_labels = BasisInfo(system, system.minao_basis).basis_labels
alignment_groups = ibo_analysis._cartesian_alignment_groups
assert len(alignment_groups) == 2
labels_by_atom = {0: [], 1: []}
for iatom, orbital_indices, target_rows in alignment_groups:
    labels_by_atom[iatom].extend(minao_labels[row].label() for row in target_rows)
    aligned_block = C_ibo_iao[np.ix_(target_rows, orbital_indices)]
    # At the Procrustes optimum this overlap is symmetric positive definite.
    np.testing.assert_allclose(aligned_block, aligned_block.T, atol=1.0e-10)
    assert np.all(np.linalg.eigvalsh(aligned_block) > 0.9)
for labels in labels_by_atom.values():
    assert labels == ["2s", "2px", "2py", "2pz"]

print(f"Active orbital indices: {active.tolist()}")
print(f"Inactive orbitals verified unchanged: {inactive.size}")
print("Active-space IBO rotation verified unitary and nontrivial.")
print("Two complete atom-local valence blocks verified aligned with 2s/2px/2py/2pz IAOs.")


IBO converged after 10 iterations.
Aligned 2 atom-local IBO block(s) to the global axis-oriented IAOs.
Atomic alignment change in IBO objective: -2.364e-09.
Active orbital indices: [2, 3, 4, 5, 6, 7, 8, 9]
Inactive orbitals verified unchanged: 20
Active-space IBO rotation verified unitary and nontrivial.
Two complete atom-local valence blocks verified aligned with 2s/2px/2py/2pz IAOs.


## Generate cube files for all orbitals

The files are written beneath the notebook's current working directory in `n2_cas66_ibo_cubes/`, with names such as `n2_cas66_ibo_00.cube`. Re-running the cell overwrites files with the same names.

In [ ]:
cube_dir = Path("n2_cas66_ibo_cubes")
n_orbitals = C_ibo.shape[1]
write_orbital_cubes(
    system=system,
    C=C_ibo,
    indices=list(range(n_orbitals)),
    prefix="n2_cas66_ibo",
    filepath=cube_dir,
)

cube_files = sorted(cube_dir.glob("n2_cas66_ibo_*.cube"))
assert len(cube_files) == n_orbitals
assert all(path.stat().st_size > 0 for path in cube_files)

print(f"Generated {len(cube_files)} cube files in {cube_dir.resolve()}")
print(f"First file: {cube_files[0].name}")
print(f"Last file:  {cube_files[-1].name}")


Generating cube files with the following parameters:
  Grid origin: (-4.000, -4.000, -4.000)
  Grid points: 40 x 40 x 59 points.
  Scaled axes: [(0.2, 0, 0), (0, 0.2, 0), (0, 0, 0.2)]
  Orbitals: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]

Generated 28 cube files in /Users/fevange/Source/forte2-dev-2/tutorials/n2_cas66_ibo_cubes
First file: n2_cas66_ibo_00.cube
Last file:  n2_cas66_ibo_27.cube
